# 🔍 Diagnóstico Operacional — GPS, Bilhetagem e GTFS

> Análise integrada do desempenho operacional do transporte público,
> combinando dados de GPS (rastreamento da frota), bilhetagem eletrônica
> (smart card) e quadro de horários (GTFS).

O notebook está organizado em dois blocos:

| Bloco | Dados | Produtos |
|---|---|---|
| **I. GPS + Bilhetagem** | Rastros GPS, registros de bilhetagem | Perfis de velocidade, VKT observada, IPK, demanda por linha/hora |
| **II. Integração com GTFS** | Bloco I + GTFS | Aderência ao quadro de horários, pontualidade, atrasos por trecho |

### Fontes de dados esperadas

| Dado | Formato típico | Colunas mínimas |
|---|---|---|
| **GPS** | CSV/Parquet | `datahora`, `latitude`, `longitude`, `veiculo`, `linha`, `sentido` |
| **Bilhetagem** | CSV/Parquet | `datahora`, `linha`, `tipo_pagamento`, `sentido` |
| **GTFS** | .zip | Padrão GTFS (stops, stop_times, trips, routes, shapes, calendar) |

---
## 0 — Configuração

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import shapely.geometry as sg
from IPython.display import display, HTML
from scipy.spatial import cKDTree

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", "{:.2f}".format)

plt.rcParams.update({
    "figure.figsize": (13, 5),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

PAL = {
    "primary": "#1565C0", "secondary": "#E65100",
    "accent": "#2E7D32", "warning": "#F9A825",
    "ida": "#1565C0", "volta": "#E65100",
}
WINDOW_COLORS = ["#455A64", "#1565C0", "#2E7D32", "#E65100", "#6A1B9A"]

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────
GPS_PATH       = "gps.parquet"          # or .csv
BILHETAGEM_PATH = "bilhetagem.parquet"  # or .csv
GTFS_PATH      = "gtfs.zip"

# ── Analysis parameters ──────────────────────────────────────────────
ANALYSIS_DATE = "2025-07-15"            # reference date (YYYY-MM-DD)
LOCAL_CRS     = 31983                   # SIRGAS 2000 / UTM 23S (adjust to your city)

TIME_WINDOWS = [
    ("Madrugada",   "00:00", "05:00"),
    ("Manhã Pico",  "05:00", "09:00"),
    ("Entrepico",   "09:00", "16:00"),
    ("Tarde Pico",  "16:00", "20:00"),
    ("Noite",       "20:00", "24:00"),
]

# GPS quality thresholds
GPS_MIN_INTERVAL_S = 10    # minimum seconds between pings
GPS_MAX_INTERVAL_S = 300   # maximum gap before splitting trip
GPS_MIN_SPEED_KMH  = 0     # filter stationary pings
GPS_MAX_SPEED_KMH  = 100   # filter GPS jumps

# IPK reference values (NTU 2024)
IPK_REF_LOW  = 0.8   # below = severe underutilisation
IPK_REF_MED  = 1.5   # typical medium city
IPK_REF_HIGH = 2.5   # above = overcrowding risk

---
---
# Bloco I — GPS e Bilhetagem (standalone)

Análises que dependem exclusivamente dos dados operacionais de GPS e
bilhetagem, sem referência ao GTFS.

## 1 — Carregamento e Auditoria do GPS

In [ ]:
# ── Load GPS ─────────────────────────────────────────────────────────
# Adapt column names to your data. The standard Brazilian format
# typically includes: datahora, latitude, longitude, veiculo, linha, sentido

if GPS_PATH.endswith(".parquet"):
    gps_raw = pd.read_parquet(GPS_PATH)
else:
    gps_raw = pd.read_csv(GPS_PATH, parse_dates=["datahora"])

# ── Standardise column names ─────────────────────────────────────────
# Map your actual column names here
COL_MAP = {
    # "your_column_name": "standard_name",
    # "dataHora": "datahora",
    # "lat": "latitude",
    # "lon": "longitude",
    # "idVeiculo": "veiculo",
    # "idLinha": "linha",
    # "sentidoViagem": "sentido",
}
if COL_MAP:
    gps_raw = gps_raw.rename(columns=COL_MAP)

# Ensure datetime
gps_raw["datahora"] = pd.to_datetime(gps_raw["datahora"])

# Filter to analysis date
gps = gps_raw.loc[gps_raw["datahora"].dt.date == pd.Timestamp(ANALYSIS_DATE).date()].copy()
gps = gps.sort_values(["veiculo", "datahora"]).reset_index(drop=True)

print(f"GPS records loaded: {len(gps_raw):,} total, {len(gps):,} on {ANALYSIS_DATE}")
print(f"Vehicles: {gps['veiculo'].nunique()}")
print(f"Lines:    {gps['linha'].nunique()}")
print(f"Span:     {gps['datahora'].min()} — {gps['datahora'].max()}")

In [ ]:
# ── Quality audit ────────────────────────────────────────────────────
gps["dt_s"] = gps.groupby("veiculo")["datahora"].diff().dt.total_seconds()

audit = {
    "Registros totais": f"{len(gps):,}",
    "Veículos": gps["veiculo"].nunique(),
    "Linhas": gps["linha"].nunique(),
    "Intervalo mediano (s)": f"{gps['dt_s'].median():.0f}",
    "Intervalo P95 (s)": f"{gps['dt_s'].quantile(0.95):.0f}",
    "Coords nulas": gps[["latitude","longitude"]].isna().any(axis=1).sum(),
    "Registros duplicados": gps.duplicated(subset=["veiculo","datahora"]).sum(),
}

display(
    pd.DataFrame(list(audit.items()), columns=["Verificação", "Valor"])
    .style.hide(axis="index").set_caption("Auditoria do GPS")
)

In [ ]:
# ── Ping frequency distribution ──────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

dt_clean = gps["dt_s"].dropna()
dt_clean = dt_clean[(dt_clean > 0) & (dt_clean < 600)]

ax1.hist(dt_clean, bins=60, color=PAL["primary"], edgecolor="white", alpha=0.8)
ax1.axvline(dt_clean.median(), color=PAL["secondary"], linestyle="--", linewidth=2,
            label=f"Mediana: {dt_clean.median():.0f}s")
ax1.set_xlabel("Intervalo entre pings (s)")
ax1.set_ylabel("Frequência")
ax1.set_title("Distribuição do intervalo GPS", fontweight="bold")
ax1.legend(frameon=False)

# Pings per hour
gps["hora"] = gps["datahora"].dt.hour
hourly_pings = gps.groupby("hora").size()
ax2.bar(hourly_pings.index, hourly_pings.values, color=PAL["accent"], edgecolor="white")
ax2.set_xlabel("Hora")
ax2.set_ylabel("Nº de pings")
ax2.set_title("Volume de pings GPS por hora", fontweight="bold")

plt.tight_layout()
plt.show()

## 2 — Reconstrução de Viagens e Cálculo de VKT

Cada sequência contínua de pings de um veículo em uma mesma linha/sentido
constitui uma viagem. Gaps maiores que `GPS_MAX_INTERVAL_S` (5 min)
dividem a sequência.

In [ ]:
# ── Compute inter-ping distances ─────────────────────────────────────
gps_proj = gpd.GeoDataFrame(
    gps,
    geometry=gpd.points_from_xy(gps.longitude, gps.latitude),
    crs=4326,
).to_crs(LOCAL_CRS)

gps["x"] = gps_proj.geometry.x
gps["y"] = gps_proj.geometry.y

# Distance to previous ping (metres)
gps["dx"] = gps.groupby("veiculo")["x"].diff()
gps["dy"] = gps.groupby("veiculo")["y"].diff()
gps["dist_m"] = np.sqrt(gps["dx"]**2 + gps["dy"]**2)

# Instantaneous speed (km/h)
gps["speed_kmh"] = np.where(
    gps["dt_s"] > 0,
    (gps["dist_m"] / gps["dt_s"]) * 3.6,
    np.nan,
)

In [ ]:
# ── Trip segmentation ────────────────────────────────────────────────
# A new trip starts when:
# - vehicle changes line or direction
# - gap > GPS_MAX_INTERVAL_S
# - speed > GPS_MAX_SPEED_KMH (GPS jump)

gps["new_trip"] = (
    (gps["veiculo"] != gps["veiculo"].shift())
    | (gps["linha"] != gps["linha"].shift())
    | (gps["sentido"] != gps["sentido"].shift())
    | (gps["dt_s"] > GPS_MAX_INTERVAL_S)
    | (gps["speed_kmh"] > GPS_MAX_SPEED_KMH)
)
gps["trip_id"] = gps["new_trip"].cumsum()

# Aggregate per trip
trips = (
    gps.groupby("trip_id")
    .agg(
        veiculo=("veiculo", "first"),
        linha=("linha", "first"),
        sentido=("sentido", "first"),
        t_inicio=("datahora", "min"),
        t_fim=("datahora", "max"),
        n_pings=("datahora", "count"),
        dist_km=("dist_m", lambda x: x.clip(lower=0).sum() / 1000),
    )
    .reset_index()
)

# Duration and speed
trips["duracao_min"] = (trips["t_fim"] - trips["t_inicio"]).dt.total_seconds() / 60
trips["vel_media_kmh"] = np.where(
    trips["duracao_min"] > 0,
    trips["dist_km"] / (trips["duracao_min"] / 60),
    np.nan,
)

# Filter: at least 3 pings, > 0.5 km, reasonable speed
trips_valid = trips.loc[
    (trips["n_pings"] >= 3)
    & (trips["dist_km"] > 0.5)
    & (trips["vel_media_kmh"].between(GPS_MIN_SPEED_KMH + 1, GPS_MAX_SPEED_KMH))
].copy()

print(f"Viagens reconstruídas: {len(trips):,} total, {len(trips_valid):,} válidas")
print(f"VKT total (válidas): {trips_valid['dist_km'].sum():,.0f} km")
print(f"Velocidade média: {trips_valid['vel_media_kmh'].median():.1f} km/h (mediana)")

In [ ]:
# ── Trip-level summary table ─────────────────────────────────────────
trip_summary = (
    trips_valid
    .groupby("linha")
    .agg(
        viagens=("trip_id", "count"),
        vkt_km=("dist_km", "sum"),
        dist_media_km=("dist_km", "mean"),
        duracao_media_min=("duracao_min", "mean"),
        vel_media_kmh=("vel_media_kmh", "mean"),
    )
    .sort_values("vkt_km", ascending=False)
    .reset_index()
)

display(
    trip_summary.head(20)
    .style
    .format({
        "vkt_km": "{:,.0f}",
        "dist_media_km": "{:.1f}",
        "duracao_media_min": "{:.0f}",
        "vel_media_kmh": "{:.1f}",
    })
    .background_gradient(subset=["vkt_km"], cmap="Blues")
    .hide(axis="index")
    .set_caption(f"VKT observada por linha — {ANALYSIS_DATE}")
)

## 3 — Perfis de Velocidade

Velocidade média observada ao longo do dia e distribuição espacial.

In [ ]:
# ── Speed by time of day ─────────────────────────────────────────────
gps_clean = gps.loc[gps["speed_kmh"].between(1, GPS_MAX_SPEED_KMH)].copy()
gps_clean["hora_dec"] = (
    gps_clean["datahora"].dt.hour
    + gps_clean["datahora"].dt.minute / 60
)

# Bin into 15-min intervals
gps_clean["bin_15"] = (gps_clean["hora_dec"] * 4).astype(int) / 4

speed_profile = (
    gps_clean.groupby("bin_15")["speed_kmh"]
    .agg(["median", "mean", "std", "count"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(
    speed_profile["bin_15"],
    speed_profile["median"] - speed_profile["std"],
    speed_profile["median"] + speed_profile["std"],
    alpha=0.15, color=PAL["primary"],
)
ax.plot(speed_profile["bin_15"], speed_profile["median"],
        color=PAL["primary"], linewidth=2, label="Mediana")
ax.plot(speed_profile["bin_15"], speed_profile["mean"],
        color=PAL["secondary"], linewidth=1.5, linestyle="--", label="Média")

# Time window shading
for i, (name, t0, t1) in enumerate(TIME_WINDOWS):
    h0, h1 = int(t0.split(":")[0]), int(t1.split(":")[0])
    ax.axvspan(h0, h1, alpha=0.04, color=WINDOW_COLORS[i])

ax.set_xlabel("Hora do dia")
ax.set_ylabel("Velocidade (km/h)")
ax.set_title("Perfil de velocidade ao longo do dia (frota inteira)", fontweight="bold")
ax.set_xlim(4, 24)
ax.xaxis.set_major_formatter(lambda x, _: f"{int(x)}h")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Speed heatmap: hour × line ───────────────────────────────────────
gps_with_line = gps_clean.merge(
    trips_valid[["trip_id", "linha"]].drop_duplicates(),
    on="trip_id", suffixes=("", "_trip"),
    how="inner",
)
# Use the trip-level linha if GPS-level is unreliable
if "linha_trip" in gps_with_line.columns:
    gps_with_line["linha"] = gps_with_line["linha_trip"]

speed_heatmap = (
    gps_with_line
    .groupby(["linha", gps_with_line["datahora"].dt.hour])["speed_kmh"]
    .median()
    .unstack()
)
# Top 15 lines by trip count
top_lines = trips_valid["linha"].value_counts().head(15).index
speed_hm_top = speed_heatmap.loc[speed_heatmap.index.isin(top_lines)]

fig = px.imshow(
    speed_hm_top,
    labels=dict(x="Hora", y="Linha", color="Vel (km/h)"),
    color_continuous_scale="RdYlGn",
    title="Velocidade mediana por linha × hora",
    aspect="auto",
)
fig.update_layout(height=450)
fig.show()

## 4 — Mapa de Velocidades por Corredor

In [ ]:
# ── Grid-based speed map ─────────────────────────────────────────────
# Bin GPS pings into a spatial grid and compute median speed per cell

from shapely.geometry import box

gps_geo = gps_clean.loc[gps_clean["speed_kmh"] > 1].copy()
gps_gdf = gpd.GeoDataFrame(
    gps_geo,
    geometry=gpd.points_from_xy(gps_geo.longitude, gps_geo.latitude),
    crs=4326,
).to_crs(LOCAL_CRS)

# Create 200m grid
bounds = gps_gdf.total_bounds
cell = 200  # metres
xs = np.arange(bounds[0], bounds[2], cell)
ys = np.arange(bounds[1], bounds[3], cell)

grid_cells = []
for x in xs:
    for y in ys:
        grid_cells.append(box(x, y, x + cell, y + cell))

grid = gpd.GeoDataFrame(geometry=grid_cells, crs=LOCAL_CRS)
grid["grid_id"] = range(len(grid))

# Spatial join
joined = gpd.sjoin(gps_gdf[["speed_kmh", "geometry"]], grid, how="inner")
grid_speed = (
    joined.groupby("grid_id")["speed_kmh"]
    .agg(["median", "count"])
    .reset_index()
)
grid_speed = grid_speed.loc[grid_speed["count"] >= 10]  # min pings per cell

grid_viz = grid.merge(grid_speed, on="grid_id").to_crs(4326)
grid_viz["centroid_lat"] = grid_viz.geometry.centroid.y
grid_viz["centroid_lon"] = grid_viz.geometry.centroid.x

fig = px.scatter_mapbox(
    grid_viz,
    lat="centroid_lat", lon="centroid_lon",
    color="median",
    size="count",
    color_continuous_scale="RdYlGn",
    size_max=8,
    mapbox_style="carto-positron",
    zoom=12,
    title="Velocidade mediana por célula (200m)",
    labels={"median": "Vel (km/h)", "count": "Pings"},
)
fig.update_layout(height=600, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

---
## 5 — Bilhetagem Eletrônica

Análise da demanda a partir dos registros de validação do cartão.

In [ ]:
# ── Load smart card data ─────────────────────────────────────────────
if BILHETAGEM_PATH.endswith(".parquet"):
    bse_raw = pd.read_parquet(BILHETAGEM_PATH)
else:
    bse_raw = pd.read_csv(BILHETAGEM_PATH, parse_dates=["datahora"])

# Standardise columns (adapt to your data)
BSE_COL_MAP = {
    # "your_column": "standard_name",
    # "dataTransacao": "datahora",
    # "idLinha": "linha",
    # "tipoPagamento": "tipo_pagamento",
}
if BSE_COL_MAP:
    bse_raw = bse_raw.rename(columns=BSE_COL_MAP)

bse_raw["datahora"] = pd.to_datetime(bse_raw["datahora"])
bse = bse_raw.loc[bse_raw["datahora"].dt.date == pd.Timestamp(ANALYSIS_DATE).date()].copy()
bse["hora"] = bse["datahora"].dt.hour

print(f"Bilhetagem: {len(bse_raw):,} total, {len(bse):,} on {ANALYSIS_DATE}")
print(f"Linhas: {bse['linha'].nunique()}")
print(f"Tipos de pagamento: {bse['tipo_pagamento'].unique().tolist()}")

In [ ]:
# ── Demand profile ───────────────────────────────────────────────────
demand_hourly = bse.groupby("hora").size()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(demand_hourly.index, demand_hourly.values,
       color=PAL["primary"], edgecolor="white", width=0.8)

# Time window shading
for i, (name, t0, t1) in enumerate(TIME_WINDOWS):
    h0, h1 = int(t0.split(":")[0]), int(t1.split(":")[0])
    ax.axvspan(h0 - 0.5, h1 - 0.5, alpha=0.06, color=WINDOW_COLORS[i])
    ax.text((h0 + h1) / 2, demand_hourly.max() * 0.97, name,
            ha="center", fontsize=8, color=WINDOW_COLORS[i], fontstyle="italic")

ax.set_xlabel("Hora")
ax.set_ylabel("Validações")
ax.set_title(f"Demanda ao longo do dia — {ANALYSIS_DATE}", fontweight="bold")
ax.xaxis.set_major_formatter(lambda x, _: f"{int(x)}h")
plt.tight_layout()
plt.show()

In [ ]:
# ── Demand by payment type ───────────────────────────────────────────
pay_type = bse["tipo_pagamento"].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors_pie = [PAL["primary"], PAL["secondary"], PAL["accent"],
              PAL["warning"], "#9C27B0", "#795548"]
ax1.pie(pay_type.values, labels=pay_type.index, autopct="%1.1f%%",
        colors=colors_pie[:len(pay_type)], startangle=90)
ax1.set_title("Composição por tipo de pagamento", fontweight="bold")

# Stacked bar by hour
pay_hourly = bse.groupby(["hora", "tipo_pagamento"]).size().unstack(fill_value=0)
pay_hourly.plot(kind="bar", stacked=True, ax=ax2, color=colors_pie, edgecolor="white", width=0.8)
ax2.set_xlabel("Hora")
ax2.set_ylabel("Validações")
ax2.set_title("Demanda por tipo × hora", fontweight="bold")
ax2.legend(fontsize=8, frameon=False, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# ── Demand by line (top 20) ──────────────────────────────────────────
demand_line = bse.groupby("linha").size().sort_values(ascending=False)

display(
    pd.DataFrame({
        "Linha": demand_line.index[:20],
        "Passageiros": demand_line.values[:20],
        "% do total": (demand_line.values[:20] / demand_line.sum() * 100),
        "% acumulado": (demand_line.values[:20] / demand_line.sum() * 100).cumsum(),
    })
    .style
    .format({"% do total": "{:.1f}%", "% acumulado": "{:.1f}%"})
    .background_gradient(subset=["Passageiros"], cmap="Oranges")
    .hide(axis="index")
    .set_caption("Top 20 linhas por demanda")
)

## 6 — Índice de Passageiros por Quilômetro (IPK)

$$\text{IPK} = \frac{\text{Passageiros transportados}}{\text{Quilômetros percorridos}}$$

Referências NTU (2024): IPK < 0.8 = subutilização severa;
1.0–1.8 = típico cidades médias; > 2.5 = superlotação.

In [ ]:
# ── Compute IPK per line ─────────────────────────────────────────────
pax_by_line = bse.groupby("linha").size().rename("passageiros")
vkt_by_line = trips_valid.groupby("linha")["dist_km"].sum().rename("vkt_km")

ipk_df = pd.DataFrame({"passageiros": pax_by_line, "vkt_km": vkt_by_line}).dropna()
ipk_df["ipk"] = ipk_df["passageiros"] / ipk_df["vkt_km"]
ipk_df = ipk_df.sort_values("ipk", ascending=False).reset_index()

# System-wide IPK
ipk_sistema = ipk_df["passageiros"].sum() / ipk_df["vkt_km"].sum()
print(f"IPK do sistema: {ipk_sistema:.2f} pass/km")

display(
    ipk_df.head(20)
    .style
    .format({"passageiros": "{:,}", "vkt_km": "{:,.0f}", "ipk": "{:.2f}"})
    .applymap(
        lambda v: f"background-color: {'#FFCDD2' if v < IPK_REF_LOW else '#C8E6C9' if v > IPK_REF_MED else ''}",
        subset=["ipk"],
    )
    .hide(axis="index")
    .set_caption(f"IPK por linha — {ANALYSIS_DATE}")
)

In [ ]:
# ── IPK distribution ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(ipk_df["ipk"].clip(upper=5), bins=30, color=PAL["primary"],
        edgecolor="white", alpha=0.8)
ax.axvline(ipk_sistema, color=PAL["secondary"], linewidth=2.5, linestyle="--",
           label=f"Sistema: {ipk_sistema:.2f}")
ax.axvline(IPK_REF_LOW, color="red", linewidth=1, linestyle=":",
           label=f"Ref. baixa: {IPK_REF_LOW}")
ax.axvline(IPK_REF_MED, color=PAL["accent"], linewidth=1, linestyle=":",
           label=f"Ref. média: {IPK_REF_MED}")

ax.set_xlabel("IPK (pass/km)")
ax.set_ylabel("Nº de linhas")
ax.set_title("Distribuição do IPK por linha", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

n_low = (ipk_df["ipk"] < IPK_REF_LOW).sum()
n_high = (ipk_df["ipk"] > IPK_REF_HIGH).sum()
print(f"Linhas com IPK < {IPK_REF_LOW}: {n_low} ({n_low/len(ipk_df)*100:.0f}%)")
print(f"Linhas com IPK > {IPK_REF_HIGH}: {n_high} ({n_high/len(ipk_df)*100:.0f}%)")

In [ ]:
# ── IPK by time window ──────────────────────────────────────────────
def assign_window(hour):
    for name, t0, t1 in TIME_WINDOWS:
        if int(t0.split(":")[0]) <= hour < int(t1.split(":")[0]):
            return name
    return "Outro"

bse["window"] = bse["hora"].apply(assign_window)
trips_valid["hora_inicio"] = trips_valid["t_inicio"].dt.hour
trips_valid["window"] = trips_valid["hora_inicio"].apply(assign_window)

pax_window = bse.groupby("window").size()
vkt_window = trips_valid.groupby("window")["dist_km"].sum()

ipk_window = pd.DataFrame({"pax": pax_window, "vkt": vkt_window}).dropna()
ipk_window["ipk"] = ipk_window["pax"] / ipk_window["vkt"]
ipk_window = ipk_window.reindex([w[0] for w in TIME_WINDOWS]).dropna()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(ipk_window)), ipk_window["ipk"],
              color=WINDOW_COLORS[:len(ipk_window)], edgecolor="white", width=0.6)

ax.axhline(ipk_sistema, color=PAL["secondary"], linestyle="--", linewidth=1.5,
           label=f"Média: {ipk_sistema:.2f}")
ax.axhline(IPK_REF_LOW, color="red", linestyle=":", alpha=0.5)

for i, (_, row) in enumerate(ipk_window.iterrows()):
    ax.text(i, row["ipk"] + 0.05, f"{row['ipk']:.2f}",
            ha="center", fontsize=10, fontweight="bold")

ax.set_xticks(range(len(ipk_window)))
ax.set_xticklabels(ipk_window.index, rotation=30, ha="right")
ax.set_ylabel("IPK (pass/km)")
ax.set_title("IPK por faixa horária", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

---
---
# Bloco II — Integração com GTFS

Cruzamento dos dados operacionais (GPS + bilhetagem) com o quadro de
horários (GTFS) para análise de aderência e pontualidade.

> Método baseado em Braga, Loureiro & Pereira (2020, 2023):
> reconstrução de horários observados de passagem em parada a partir
> do GPS, seguida de comparação com os horários programados do GTFS.

## 7 — Carregamento do GTFS

In [ ]:
import gtfs_kit as gk

feed = gk.read_feed(GTFS_PATH, dist_units="km")
if "shape_dist_traveled" not in feed.stop_times.columns:
    feed = feed.append_dist_to_stop_times()

# Active trips on analysis date
gtfs_date = ANALYSIS_DATE.replace("-", "")
activity = feed.compute_trip_activity([gtfs_date])
active_trip_ids = set(activity.loc[activity[gtfs_date] > 0, "trip_id"])

gtfs_trips = feed.trips.loc[feed.trips["trip_id"].isin(active_trip_ids)].copy()
gtfs_st = feed.stop_times.loc[feed.stop_times["trip_id"].isin(active_trip_ids)].copy()

print(f"GTFS: {len(gtfs_trips)} viagens ativas em {gtfs_date}")
print(f"      {len(gtfs_st)} stop_times entries")
print(f"      {feed.stops['stop_id'].nunique()} paradas")

## 8 — Estimativa de Horários de Passagem em Parada

Para cada viagem GPS, projeta-se os pings sobre a geometria do shape
correspondente e interpola-se o instante de passagem em cada parada.

In [ ]:
# ── Build stop spatial index ─────────────────────────────────────────
stops_gdf = gpd.GeoDataFrame(
    feed.stops,
    geometry=gpd.points_from_xy(feed.stops.stop_lon, feed.stops.stop_lat),
    crs=4326,
).to_crs(LOCAL_CRS)

stop_coords = np.array(list(zip(stops_gdf.geometry.x, stops_gdf.geometry.y)))
stop_tree = cKDTree(stop_coords)
stop_ids = stops_gdf["stop_id"].values

SNAP_RADIUS = 50  # metres

In [ ]:
def estimate_stop_arrivals(
    trip_gps: pd.DataFrame,
    stop_tree: cKDTree,
    stop_ids: np.ndarray,
    snap_radius: float = SNAP_RADIUS,
) -> pd.DataFrame:
    """
    For a single GPS trip, find the closest timestamp to each nearby stop.

    Returns DataFrame with stop_id, estimated_arrival (datetime).
    """
    coords = np.array(list(zip(trip_gps["x"], trip_gps["y"])))
    times = trip_gps["datahora"].values

    # Query all GPS points against all stops
    dists, idxs = stop_tree.query(coords)
    mask = dists <= snap_radius

    if not mask.any():
        return pd.DataFrame(columns=["stop_id", "estimated_arrival", "snap_dist_m"])

    rows = []
    matched = pd.DataFrame({
        "stop_id": stop_ids[idxs[mask]],
        "time": times[mask],
        "dist": dists[mask],
    })

    # For each stop, take the ping with minimum distance
    for sid, grp in matched.groupby("stop_id"):
        best = grp.loc[grp["dist"].idxmin()]
        rows.append({
            "stop_id": sid,
            "estimated_arrival": pd.Timestamp(best["time"]),
            "snap_dist_m": best["dist"],
        })

    return pd.DataFrame(rows)

In [ ]:
# ── Run stop arrival estimation for all GPS trips ────────────────────
# This can be slow for large datasets — consider sampling or parallelising

from tqdm.auto import tqdm

arrivals_list = []
sample_trips = trips_valid.head(500)  # adjust sample size as needed

for _, trip in tqdm(sample_trips.iterrows(), total=len(sample_trips), desc="Estimating arrivals"):
    trip_gps = gps.loc[gps["trip_id"] == trip["trip_id"]]
    if len(trip_gps) < 3:
        continue
    est = estimate_stop_arrivals(trip_gps, stop_tree, stop_ids)
    if not est.empty:
        est["gps_trip_id"] = trip["trip_id"]
        est["linha"] = trip["linha"]
        est["sentido"] = trip["sentido"]
        arrivals_list.append(est)

arrivals = pd.concat(arrivals_list, ignore_index=True) if arrivals_list else pd.DataFrame()
print(f"Stop arrivals estimated: {len(arrivals):,} across {arrivals['gps_trip_id'].nunique()} trips")

## 9 — Associação ao Quadro de Horários

Cada viagem GPS é pareada com a viagem GTFS mais próxima em termos de
linha, sentido e horário de partida. O desvio entre o horário estimado
e o programado constitui o **indicador de pontualidade**.

In [ ]:
# ── Match GPS trips to GTFS trips ────────────────────────────────────
# Strategy: for each GPS trip, find the GTFS trip on the same route
# with the closest departure time.

def hhmmss_to_seconds(t: str) -> int:
    parts = t.split(":")
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

# Build GTFS trip departures
gtfs_first_stop = (
    gtfs_st.sort_values(["trip_id", "stop_sequence"])
    .groupby("trip_id")
    .first()
    .reset_index()
    [["trip_id", "arrival_time", "stop_id"]]
)
gtfs_first_stop["dep_seconds"] = gtfs_first_stop["arrival_time"].apply(hhmmss_to_seconds)

# Merge with route info
gtfs_first_stop = gtfs_first_stop.merge(
    gtfs_trips[["trip_id", "route_id", "direction_id"]],
    on="trip_id",
)
gtfs_first_stop = gtfs_first_stop.merge(
    feed.routes[["route_id", "route_short_name"]],
    on="route_id",
)

# GPS trip departures
gps_deps = trips_valid[["trip_id", "linha", "sentido", "t_inicio"]].copy()
gps_deps["dep_seconds"] = (
    gps_deps["t_inicio"].dt.hour * 3600
    + gps_deps["t_inicio"].dt.minute * 60
    + gps_deps["t_inicio"].dt.second
)
gps_deps = gps_deps.rename(columns={"trip_id": "gps_trip_id"})

print(f"GPS trips to match: {len(gps_deps)}")
print(f"GTFS trips available: {len(gtfs_first_stop)}")

In [ ]:
# ── Nearest-time matching ───────────────────────────────────────────
matches = []

for linha in gps_deps["linha"].unique():
    gps_sub = gps_deps.loc[gps_deps["linha"].astype(str) == str(linha)]
    gtfs_sub = gtfs_first_stop.loc[
        gtfs_first_stop["route_short_name"].astype(str) == str(linha)
    ]

    if gtfs_sub.empty:
        continue

    for _, gps_row in gps_sub.iterrows():
        diffs = (gtfs_sub["dep_seconds"] - gps_row["dep_seconds"]).abs()
        best_idx = diffs.idxmin()
        best_diff = diffs.loc[best_idx]

        if best_diff <= 3600:  # max 1 hour difference
            gtfs_row = gtfs_sub.loc[best_idx]
            matches.append({
                "gps_trip_id": gps_row["gps_trip_id"],
                "gtfs_trip_id": gtfs_row["trip_id"],
                "linha": linha,
                "gps_dep_s": gps_row["dep_seconds"],
                "gtfs_dep_s": gtfs_row["dep_seconds"],
                "dep_diff_min": best_diff / 60,
            })

match_df = pd.DataFrame(matches)
print(f"Matched: {len(match_df)} GPS trips → GTFS trips")
print(f"Match rate: {len(match_df) / len(gps_deps) * 100:.1f}%")
print(f"Median departure diff: {match_df['dep_diff_min'].median():.1f} min")

## 10 — Indicador de Pontualidade

Desvio entre o horário observado (GPS) e o programado (GTFS) em cada
parada, agregado por linha e faixa horária.

Classificação:
- **Adiantado** (> 1 min antes): pode causar perda de embarque
- **No horário** (±1 min)
- **Atrasado** (> 1 min depois): acumula espera
- **Muito atrasado** (> 5 min): serviço severamente degradado

In [ ]:
# ── Compute stop-level deviations ────────────────────────────────────
# Join estimated arrivals with GTFS scheduled arrivals

if not arrivals.empty and not match_df.empty:
    # Get GTFS scheduled times for matched trips
    sched = gtfs_st[["trip_id", "stop_id", "arrival_time", "stop_sequence"]].copy()
    sched["sched_seconds"] = sched["arrival_time"].apply(hhmmss_to_seconds)
    sched = sched.rename(columns={"trip_id": "gtfs_trip_id"})

    # Join: arrivals → match → schedule
    dev = (
        arrivals
        .merge(match_df[["gps_trip_id", "gtfs_trip_id", "linha"]], on="gps_trip_id")
        .merge(sched, on=["gtfs_trip_id", "stop_id"], how="inner")
    )

    # Deviation in minutes (positive = late)
    dev["obs_seconds"] = (
        dev["estimated_arrival"].dt.hour * 3600
        + dev["estimated_arrival"].dt.minute * 60
        + dev["estimated_arrival"].dt.second
    )
    dev["desvio_min"] = (dev["obs_seconds"] - dev["sched_seconds"]) / 60

    # Classify
    dev["status"] = pd.cut(
        dev["desvio_min"],
        bins=[-np.inf, -1, 1, 5, np.inf],
        labels=["Adiantado", "No horário", "Atrasado", "Muito atrasado"],
    )

    print(f"Stop-level deviations computed: {len(dev):,}")
    print(f"\nDistribuição de pontualidade:")
    print(dev["status"].value_counts().to_string())
else:
    dev = pd.DataFrame()
    print("Insufficient data for punctuality analysis")

In [ ]:
# ── Punctuality summary by line ──────────────────────────────────────
if not dev.empty:
    punct_line = (
        dev.groupby("linha")
        .agg(
            obs=("desvio_min", "count"),
            desvio_medio=("desvio_min", "mean"),
            desvio_mediano=("desvio_min", "median"),
            pct_no_horario=("status", lambda x: (x == "No horário").mean() * 100),
            pct_atrasado=("status", lambda x: x.isin(["Atrasado","Muito atrasado"]).mean() * 100),
        )
        .sort_values("pct_no_horario", ascending=True)
        .reset_index()
    )

    display(
        punct_line.head(20)
        .style
        .format({
            "desvio_medio": "{:.1f}",
            "desvio_mediano": "{:.1f}",
            "pct_no_horario": "{:.0f}%",
            "pct_atrasado": "{:.0f}%",
        })
        .background_gradient(subset=["pct_no_horario"], cmap="RdYlGn")
        .background_gradient(subset=["pct_atrasado"], cmap="YlOrRd")
        .hide(axis="index")
        .set_caption("Pontualidade por linha (% no horário)")
    )

In [ ]:
# ── Deviation histogram ──────────────────────────────────────────────
if not dev.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    d = dev["desvio_min"].clip(-15, 15)

    ax.hist(d, bins=60, color=PAL["primary"], edgecolor="white", alpha=0.8)
    ax.axvline(0, color="black", linewidth=1.5)
    ax.axvspan(-1, 1, alpha=0.15, color=PAL["accent"], label="No horário (±1 min)")
    ax.axvline(d.median(), color=PAL["secondary"], linestyle="--", linewidth=2,
               label=f"Mediana: {d.median():.1f} min")

    ax.set_xlabel("Desvio (min) — negativo = adiantado, positivo = atrasado")
    ax.set_ylabel("Frequência")
    ax.set_title("Distribuição do desvio em relação ao quadro de horários", fontweight="bold")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    # Global punctuality index
    ip = (dev["status"] == "No horário").mean() * 100
    print(f"\nÍndice de Pontualidade global: {ip:.1f}%")
    print(f"  Adiantado:      {(dev['status']=='Adiantado').mean()*100:.1f}%")
    print(f"  No horário:     {(dev['status']=='No horário').mean()*100:.1f}%")
    print(f"  Atrasado:       {(dev['status']=='Atrasado').mean()*100:.1f}%")
    print(f"  Muito atrasado: {(dev['status']=='Muito atrasado').mean()*100:.1f}%")

In [ ]:
# ── Punctuality by time of day ───────────────────────────────────────
if not dev.empty:
    dev["hora"] = dev["estimated_arrival"].dt.hour
    dev["window"] = dev["hora"].apply(assign_window)

    punct_window = (
        dev.groupby("window")["status"]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex([w[0] for w in TIME_WINDOWS])
        .dropna(how="all")
        * 100
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    punct_window.plot(
        kind="bar", stacked=True, ax=ax,
        color=["#2196F3", "#4CAF50", "#FF9800", "#F44336"],
        edgecolor="white", width=0.7,
    )
    ax.set_ylabel("% das passagens")
    ax.set_xlabel("")
    ax.set_title("Pontualidade por faixa horária", fontweight="bold")
    ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1, 1))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

---

### Notas metodológicas

- **Reconstrução de viagens GPS**: sequências de pings segmentadas por
  veículo, linha, sentido e gaps temporais > 5 min.
- **Snap de paradas**: raio de 50m (KDTree); ping mais próximo por parada.
- **Associação GTFS**: matching temporal por linha (±1h max).
- **Pontualidade**: desvio observado − programado; ±1 min = no horário.
- **IPK**: passageiros validados ÷ VKT observada (GPS).
- **Referências**: Braga, Loureiro & Pereira (2020, 2023);
  NTU Anuário 2023/2024; ANTP (2013).